# Zolai Cleaning Pipeline (Local Gemma 2B)

Full workflow: load → filter → batch → prompt → parse JSON → save **`zolai_after_local_gemma.jsonl`** (rows with `text`, optional `translation_en`).

**Cooperates with:** **`zolai-dataset-combiner.ipynb`** (auto-picks `zolai_after_local_gemma.jsonl` from `/kaggle/working`) → merged JSONL + optional HF export → **`zolai-qwen-training-v2.ipynb`** via `ZOLOAI_DATASET_SRC` or working `zolai_hf_disk_export`.

For **Gemini API** cleaning in one notebook, use **`zolai-cleaner-v2.ipynb`** (`RUN_GEMINI_REFINE=True`).

Optimized for Kaggle T4 GPU; enable Internet for `pip` and Hugging Face model download.

In [1]:
!pip install transformers accelerate bitsandbytes -q

In [2]:
import json
import os
from pathlib import Path

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from tqdm import tqdm

IS_KAGGLE = Path("/kaggle").exists()
WORK_DIR = Path("/kaggle/working" if IS_KAGGLE else os.environ.get("ZOLOAI_WORK_DIR", ".")).resolve()
DATASETS_ROOT = Path(os.environ.get("ZOLOAI_DATASETS_ROOT", "/kaggle/input"))

def discover_inputs(root: Path) -> list[Path]:
    candidates = []
    if root.exists():
        candidates.extend(list(root.rglob("*.jsonl")))
    for extra in (WORK_DIR / "zolai_cleaner_out", WORK_DIR):
        if extra.is_dir():
            candidates.extend(list(extra.glob("*.jsonl")))
    
    # Priority: processed cleaner files first
    def priority(p: Path):
        n = p.name.lower()
        if "after_gemini" in n: return 0
        if "gold_train" in n: return 1
        if "train_ready" in n: return 2
        if "train" in n: return 3
        return 10
    
    return sorted(set(candidates), key=lambda x: (priority(x), x.name))

cands = discover_inputs(DATASETS_ROOT)
INPUT_PATH = cands[0] if cands else None
print("INPUT_PATH:", INPUT_PATH)


## Step 1: Load & Filter Dataset

In [4]:
def load_data(path, limit=None):
    if not path or not path.is_file():
        print(f"WARNING: Input path {path} not found.")
        return []
    data = []
    with open(path, 'r', encoding='utf-8') as f:
        for i, line in enumerate(f):
            try:
                item = json.loads(line)
                text = item.get('text')
                if isinstance(text, str) and len(text.strip()) > 5:
                    data.append(text.strip())
            except:
                continue
            if limit and len(data) >= limit:
                break
    return data

# 🔥 IMPORTANT: start small
texts = load_data(INPUT_PATH)
print('Loaded:', len(texts))

Loaded: 2109286


## Step 2: Load Gemma Model (4bit)

In [11]:
import os

from kaggle_secrets import UserSecretsClient
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

# 2× T4: use both GPUs by default (device_map spreads layers). ZOLOAI_USE_ONE_GPU=1 → GPU 0 only.
def _env_truthy(name: str) -> bool:
    v = os.environ.get(name, "").strip().lower()
    return v in ("1", "true", "yes")


_one_gpu = _env_truthy("ZOLOAI_USE_ONE_GPU")
if not _one_gpu:
    try:
        _sec = UserSecretsClient().get_secret("ZOLOAI_USE_ONE_GPU")
        if _sec and str(_sec).strip().lower() in ("1", "true", "yes"):
            _one_gpu = True
    except Exception:
        pass

if os.path.isdir("/kaggle") and _one_gpu:
    os.environ["CUDA_VISIBLE_DEVICES"] = "0"
    print("ZOLOAI_USE_ONE_GPU: GPU 0 only")
else:
    print("GPU mode: all GPUs visible (2× T4 supported)")

print("🔹 Loading HuggingFace token...")
user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF_TOKEN")

MODEL_NAME = "google/gemma-2-2b-it"
print(f"🔹 Model: {MODEL_NAME}")

# Quantization config
print("🔹 Setting up 4-bit quantization...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

# Tokenizer
print("🔹 Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    token=HF_TOKEN
)

# Model
print("🔹 Loading model (this may take time)...")
_max_memory = None
if torch.cuda.is_available() and torch.cuda.device_count() >= 2:
    _max_memory = {i: "14GiB" for i in range(torch.cuda.device_count())}
    print("🔹 max_memory (2× T4):", _max_memory)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    max_memory=_max_memory,
    quantization_config=bnb_config,
    token=HF_TOKEN
)

print("✅ Model loaded successfully!")

# GPU info
print("🔹 Device map:", model.hf_device_map)

Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

Model loaded!


## Step 3: Prompt Builder

In [1]:
def build_prompt(batch):
    joined = "\n".join([f"{i+1}. {x}" for i, x in enumerate(batch)])
    return (
        "You are a Zolai (Tedim dialect) linguist and dataset cleaner.\n"
        "Return ONLY valid JSON (no markdown, no extra text).\n\n"
        "For each input sentence, output an object with keys:\n"
        "- zolai: the original input\n"
        "- fixed: corrected Zolai (minimal edits; keep meaning; do not add content)\n"
        "- english: English translation\n\n"
        "Return a JSON array of the same length as the inputs.\n\n"
        "Inputs:\n"
        f"{joined}\n"
    )

## Step 4: Generate Function

In [ ]:
def _extract_first_json_array(s: str):
    if not s:
        return None
    start = s.find("[")
    if start < 0:
        return None
    depth = 0
    for i in range(start, len(s)):
        ch = s[i]
        if ch == "[":
            depth += 1
        elif ch == "]":
            depth -= 1
            if depth == 0:
                return s[start : i + 1]
    return None


def generate_json(prompt, max_new_tokens=512):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=0.0,
        do_sample=False,
    )
    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Many instruction-tuned models echo the prompt; try to extract the JSON array.
    raw = _extract_first_json_array(decoded) or decoded
    return json.loads(raw)


## Step 5: Batch Processing

In [ ]:
def batch_data(data, size=10):
    for i in range(0, len(data), size):
        yield data[i:i+size]

## Step 6: Cleaning Loop (with checkpoint)

In [ ]:
OUT_JSONL = "/kaggle/working/zolai_after_local_gemma.jsonl"
PROGRESS_JSON = "/kaggle/working/zolai_after_local_gemma.progress.json"
BATCH_SIZE = 10
SAVE_EVERY_BATCHES = 50

# Resume support
state = {"batch_idx": 0, "written": 0}
if os.path.isfile(PROGRESS_JSON) and os.path.isfile(OUT_JSONL):
    try:
        with open(PROGRESS_JSON, "r", encoding="utf-8") as f:
            state = json.load(f)
    except Exception:
        state = state

start_batch = int(state.get("batch_idx", 0))
written = int(state.get("written", 0))

print("Writing to:", OUT_JSONL)
print("Resuming at batch:", start_batch, "| already written:", written)

batch_idx = -1
for batch_idx, batch in enumerate(tqdm(batch_data(texts, BATCH_SIZE)), start=0):
    if batch_idx < start_batch:
        continue

    prompt = build_prompt(batch)

    try:
        items = generate_json(prompt)
        if not isinstance(items, list) or len(items) != len(batch):
            raise ValueError(f"Bad model output shape (got {type(items)} len={getattr(items,'__len__',lambda:None)()})")

        with open(OUT_JSONL, "a", encoding="utf-8") as f:
            for raw_in, it in zip(batch, items):
                if not isinstance(it, dict):
                    it = {}
                fixed = (it.get("fixed") or raw_in).strip()
                en = (it.get("english") or "").strip()
                row = {
                    "text": fixed,
                    "zolai_raw": raw_in.strip(),
                    "translation_en": en,
                    "refiner": "local_gemma",
                    "refiner_model": MODEL_NAME,
                }
                f.write(json.dumps(row, ensure_ascii=False) + "\n")
                written += 1

    except Exception as e:
        print("Error on batch", batch_idx, ":", e)

    if SAVE_EVERY_BATCHES and (batch_idx + 1) % SAVE_EVERY_BATCHES == 0:
        with open(PROGRESS_JSON, "w", encoding="utf-8") as f:
            json.dump({"batch_idx": batch_idx + 1, "written": written}, f, indent=2)

with open(PROGRESS_JSON, "w", encoding="utf-8") as f:
    json.dump({"batch_idx": batch_idx + 1, "written": written, "done": True}, f, indent=2)

print("Done. Wrote rows:", written)
print("Output:", OUT_JSONL)


## Step 7: Save Final Output

In [ ]:
print("Saved JSONL rows (already streamed during processing):", OUT_JSONL)
print("Progress file:", PROGRESS_JSON)


## Notes
- Use T4 GPU (TPU not supported)
- Start with 50k samples
- Increase gradually
- Batch size 5–10 recommended
